# nb12.4 — The Conference-Calendar Engine: Twenty Conferences, One Ranked List

*Week 12, Chapter 12.4.* You finished the paper. Now where does it go next?
Eastern Finance? FMA Doctoral Consortium? An undergraduate research symposium?
The right answer depends on your paper's topic, your status (undergraduate,
graduate, junior faculty), your travel budget, and how lucky you are willing to
feel about acceptance rates.

This notebook builds a synthetic dataset of 20 finance and economics conferences
with realistic submission deadlines, fees, acceptance rates, and undergraduate
eligibility flags. Then it scores each conference for fit against a given paper
and produces a rank-ordered shortlist. The dataset is synthetic -- generated
fresh in this notebook -- so the notebook stays reproducible offline. The
**scoring engine** is the part you will reuse with real data.


In [ ]:
import matplotlib
matplotlib.use("Agg")

from dataclasses import asdict, dataclass, field
from datetime import date, timedelta

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)


## 1. Generating the synthetic conference dataset

We hand-curate 20 conferences with names that *resemble* but do not equal real
conferences. Acceptance rates, fees, and undergraduate-eligibility flags are
drawn from plausible distributions. Submission deadlines are anchored to a
hypothetical academic calendar starting today (2026-09-11) and rolling forward.

Every conference gets a vector of *topic tags* (e.g., `corporate-finance`,
`asset-pricing`, `fair-lending`) that the scoring engine uses to match against
the paper.


In [ ]:
TODAY = date(2026, 9, 11)

# Curated list of 20 conferences (fictional but plausible)
CONF_DATA = [
    # (name, type, days_until_deadline, fee_usd, acceptance_rate, undergrad_ok, tags)
    ("Eastern Finance Association Annual Meeting",          "association",  60,  450, 0.45, False,
     ["corporate-finance", "asset-pricing", "banking"]),
    ("Financial Management Association Annual Meeting",     "association",  85,  500, 0.55, False,
     ["corporate-finance", "investments", "international"]),
    ("Midwest Finance Association Annual Meeting",          "association",  72,  400, 0.50, False,
     ["corporate-finance", "asset-pricing", "behavioral"]),
    ("Southern Finance Association Annual Meeting",         "association",  90,  425, 0.55, False,
     ["corporate-finance", "investments", "banking"]),
    ("Western Finance Association Annual Meeting",          "association", 120,  600, 0.18, False,
     ["corporate-finance", "asset-pricing", "frontier"]),
    ("AEA/ASSA Annual Meeting (poster session)",            "association", 110,  350, 0.40, True,
     ["economics", "corporate-finance", "labor"]),
    ("AFA Doctoral Tutorial",                                "doctoral",    95,  100, 0.20, False,
     ["asset-pricing", "frontier"]),
    ("FMA Doctoral Consortium",                              "doctoral",    80,    0, 0.30, False,
     ["corporate-finance", "career"]),
    ("Schar Symposium for Undergraduate Research",          "undergrad",    25,    0, 0.85, True,
     ["fair-lending", "climate-risk", "crypto", "patents", "momentum"]),
    ("National Conference on Undergraduate Research",        "undergrad",    50,  150, 0.80, True,
     ["all-fields"]),
    ("ACUFE Undergraduate Finance Conference",               "undergrad",    40,  200, 0.70, True,
     ["corporate-finance", "investments", "fintech"]),
    ("Posters on the Hill",                                  "undergrad",   100,    0, 0.10, True,
     ["policy-relevant"]),
    ("CFE Conference on Computational and Financial Econometrics", "specialty", 65, 550, 0.50, False,
     ["econometrics", "machine-learning", "asset-pricing"]),
    ("NBER Summer Institute (Asset Pricing)",                "elite",       130, 1200, 0.10, False,
     ["asset-pricing", "frontier"]),
    ("NBER Summer Institute (Corporate Finance)",            "elite",       130, 1200, 0.10, False,
     ["corporate-finance", "frontier"]),
    ("FDIC Bank Research Conference",                        "regulator",    75,    0, 0.25, False,
     ["banking", "regulation", "fair-lending"]),
    ("Federal Reserve System Conference on Climate Risk",   "regulator",   105,    0, 0.30, False,
     ["climate-risk", "banking", "regulation"]),
    ("FinTech Frontiers (Stanford GSB)",                     "specialty",    55,  650, 0.35, False,
     ["crypto", "fintech", "fair-lending"]),
    ("Conference on Empirical Legal Studies",                "interdisciplinary", 70, 400, 0.45, False,
     ["fair-lending", "law", "policy-relevant"]),
    ("USPTO Patent Statistics for Decision-Makers",          "specialty",   115,  100, 0.55, True,
     ["patents", "innovation", "policy-relevant"]),
]

conferences = pd.DataFrame(
    CONF_DATA,
    columns=["name", "type", "days_to_deadline", "fee_usd",
             "acceptance_rate", "undergrad_ok", "tags"],
)
conferences["deadline_date"] = conferences["days_to_deadline"].apply(
    lambda d: TODAY + timedelta(days=int(d))
)
conferences[["name", "type", "deadline_date", "fee_usd", "acceptance_rate", "undergrad_ok"]].head(8)


## 2. The paper-fit object

A `PaperFit` carries everything the engine needs about *your* paper to score
conferences. Topic tags must overlap with the conference's tags; the author's
status (undergraduate / graduate / faculty) interacts with the conference's
eligibility flag; budget and current date interact with fees and deadlines.


In [ ]:
@dataclass
class PaperFit:
    title: str
    author_status: str             # "undergraduate", "graduate", "faculty"
    topic_tags: list
    budget_usd: float              # total willingness to spend (registration + travel)
    travel_budget_usd: float       # assumed travel cost component
    earliest_submit_date: date     # paper not ready before this date
    prefers_undergrad_venues: bool = True

maya = PaperFit(
    title="Race, ZIP Codes, and Mortgage Denials in HMDA 2018-2023",
    author_status="undergraduate",
    topic_tags=["fair-lending", "banking", "policy-relevant"],
    budget_usd=800.0,
    travel_budget_usd=350.0,    # the registration fee budget is 800 - 350 = 450
    earliest_submit_date=TODAY,
    prefers_undergrad_venues=True,
)
asdict(maya)


## 3. The scoring engine

We score each conference on five axes, each on a 0-1 scale, then take a
weighted sum. The weights reflect what an undergraduate author actually
optimizes (eligibility > topic > timing > cost > selectivity-as-a-positive).

| Axis | Range | What it measures |
|------|-------|------------------|
| `topic_score`       | 0-1 | Jaccard overlap between paper tags and conference tags |
| `eligibility_score` | 0/1 | 1 if undergraduate-eligible (or status > undergrad) |
| `timing_score`      | 0-1 | Deadline is at least 14 days out, not more than 180 |
| `cost_score`        | 0-1 | Registration fee fits within budget minus travel reserve |
| `selectivity_pref`  | 0-1 | Higher acceptance rate = higher score for an undergrad |


In [ ]:
def jaccard(a, b):
    sa, sb = set(a), set(b)
    if not sa and not sb:
        return 0.0
    if "all-fields" in sb:
        return 0.5  # all-field venues are weakly matched to anything
    return len(sa & sb) / len(sa | sb)

def score_topic(paper, conf):
    return jaccard(paper.topic_tags, conf["tags"])

def score_eligibility(paper, conf):
    if paper.author_status == "undergraduate":
        return 1.0 if conf["undergrad_ok"] else 0.0
    return 1.0  # graduate and faculty can submit anywhere

def score_timing(paper, conf, today=TODAY):
    days = (conf["deadline_date"] - today).days
    if days < 14:
        return 0.0                # too close to write a clean submission
    if days > 180:
        return 0.4                # too far - your paper will look stale by then
    if days <= 90:
        return 1.0                # the sweet spot
    return 0.7                    # 91-180 days, still good

def score_cost(paper, conf):
    available = paper.budget_usd - paper.travel_budget_usd
    fee = conf["fee_usd"]
    if available <= 0:
        return 0.0 if fee > 0 else 1.0
    if fee == 0:
        return 1.0
    return max(0.0, 1.0 - fee / max(available, 1.0))

def score_selectivity(paper, conf):
    rate = conf["acceptance_rate"]
    if paper.author_status == "undergraduate":
        # Undergrads weight high acceptance rates favorably
        return float(rate)
    # Faculty: a *lower* acceptance rate is mildly preferred (reputation signal)
    return 1.0 - 0.5 * rate

WEIGHTS = {
    "topic_score": 0.35,
    "eligibility_score": 0.25,
    "timing_score": 0.15,
    "cost_score": 0.15,
    "selectivity_pref": 0.10,
}

def rank_conferences(paper, confs, weights=WEIGHTS, today=TODAY):
    out = confs.copy()
    out["topic_score"]       = out.apply(lambda r: score_topic(paper, r), axis=1)
    out["eligibility_score"] = out.apply(lambda r: score_eligibility(paper, r), axis=1)
    out["timing_score"]      = out.apply(lambda r: score_timing(paper, r, today), axis=1)
    out["cost_score"]        = out.apply(lambda r: score_cost(paper, r), axis=1)
    out["selectivity_pref"]  = out.apply(lambda r: score_selectivity(paper, r), axis=1)
    out["total_score"] = sum(out[k] * w for k, w in weights.items())
    return out.sort_values("total_score", ascending=False).reset_index(drop=True)

ranked = rank_conferences(maya, conferences)
ranked[["name", "topic_score", "eligibility_score", "cost_score", "total_score"]].head(8)


**Reading the ranking.** Maya's top venues are the Schar Symposium (her topic
tags are exactly in scope), Posters on the Hill (policy-relevant fair-lending),
and the FDIC Bank Research Conference (banking + fair-lending). The big
association meetings rank low for her not because the topic is wrong but because
they are not undergraduate-eligible.

**Why weights, not a hard filter.** A pure filter on `undergrad_ok` would
discard the FDIC and Empirical Legal Studies conferences -- venues that
occasionally accept exceptional undergraduate work even though they do not
advertise an undergraduate track. Soft scoring keeps them visible.


## 4. The shortlist function

Most authors want three to five venues, not twenty. We expose `shortlist(paper, n=5)`
that returns the top-n with human-readable rationale and the deadline horizon
in plain English.


In [ ]:
def days_string(days):
    if days < 0:
        return f"PAST due ({-days} days ago)"
    if days < 30:
        return f"{days} days (urgent)"
    if days < 90:
        return f"{days} days (comfortable)"
    return f"{days} days (long horizon)"

def shortlist(paper, confs, n=5, today=TODAY):
    r = rank_conferences(paper, confs, today=today).head(n)
    r["deadline_human"] = r["deadline_date"].apply(
        lambda d: days_string((d - today).days)
    )
    cols = ["name", "type", "deadline_human", "fee_usd", "acceptance_rate", "total_score"]
    return r[cols]

shortlist(maya, conferences, n=5)


## 5. A second author: Devon's stablecoin paper

To confirm the engine generalizes, we rank the same 20 conferences for Devon's
crypto-payments paper. He is also an undergraduate but his topic tags
(`crypto`, `fintech`, `banking`, `international`) overlap with different venues.
The ranking should shift accordingly.


In [ ]:
devon = PaperFit(
    title="Stablecoin Adoption and Cross-Border Remittance Fees",
    author_status="undergraduate",
    topic_tags=["crypto", "fintech", "banking", "international"],
    budget_usd=600.0,
    travel_budget_usd=300.0,
    earliest_submit_date=TODAY,
    prefers_undergrad_venues=True,
)
shortlist(devon, conferences, n=5)


## 6. Sensitivity: what if Maya could spend more?

A useful diagnostic is *how much would the ranking change if my budget went up
by $X*? The cell below sweeps Maya's budget from \$400 to \$2000 and plots the
total score of her top-3 conferences. Flat curves mean the ranking is robust
to budget. Sharply rising curves mean a venue is gated by your wallet.


In [ ]:
budgets = np.linspace(400, 2000, 17)
top3 = shortlist(maya, conferences, n=3)["name"].tolist()

curves = {name: [] for name in top3}
for B in budgets:
    sweep_paper = PaperFit(
        title=maya.title, author_status="undergraduate", topic_tags=maya.topic_tags,
        budget_usd=float(B), travel_budget_usd=maya.travel_budget_usd,
        earliest_submit_date=maya.earliest_submit_date,
    )
    r = rank_conferences(sweep_paper, conferences)
    for name in top3:
        curves[name].append(r.set_index("name").loc[name, "total_score"])

fig, ax = plt.subplots(figsize=(7, 4.5))
for name, vals in curves.items():
    short = name[:40] + ("..." if len(name) > 40 else "")
    ax.plot(budgets, vals, marker="o", label=short)
ax.set_xlabel("Total budget (USD)")
ax.set_ylabel("Total score")
ax.set_title("Sensitivity of Maya's top-3 venues to budget")
ax.legend(loc="lower right", fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig("/tmp/nb12_4_budget_sensitivity.png", dpi=120)
plt.close(fig)
print("saved sensitivity chart to /tmp/nb12_4_budget_sensitivity.png")


## 7. The deadline calendar

The other deliverable is a calendar of upcoming deadlines for the venues that
remain after scoring. We sort by deadline (not by score) and print a checklist
suitable for pasting into Notion or Google Calendar.


In [ ]:
def deadline_calendar(paper, confs, min_score=0.5, today=TODAY):
    r = rank_conferences(paper, confs, today=today)
    r = r[r["total_score"] >= min_score].copy()
    r["deadline_iso"] = r["deadline_date"].apply(lambda d: d.isoformat())
    r["days_left"] = r["deadline_date"].apply(lambda d: (d - today).days)
    return r.sort_values("deadline_date")[
        ["deadline_iso", "days_left", "name", "fee_usd", "total_score"]
    ].reset_index(drop=True)

deadline_calendar(maya, conferences, min_score=0.45)


## 8. When it fails

**Tag mismatch.** If your paper's tags do not match the catalog, the topic
score is zero across the board and the ranking collapses to eligibility + cost
+ timing. The fix is to widen your tags (e.g., add `policy-relevant` if your
fair-lending paper makes a policy recommendation) or to enrich the conference
tag list.

**Stale catalog.** A real version of this notebook reads a CSV maintained by a
human; the synthetic dataset here goes stale the day it is generated. Schar
keeps a Google Sheet of acceptable venues for symposium-prep papers; in
production you would `pd.read_csv` from that sheet.

**Selectivity preference reversal.** We assume undergraduates weight *high*
acceptance rates favorably (a 50%-acceptance regional conference is a better
practical target than the 10% WFA). A faculty version of this engine should
flip that sign. The current code does that, but the weight (0.10) is small --
a graduate student making a name should bump it up.


## 9. Your turn

1. **Enrich the catalog.** Add five more conferences relevant to *your*
   paper. Include the deadline, fee, acceptance rate, and tag list. Re-rank.

2. **Add a quality signal.** Add a column `prestige` on a 1-5 scale and a
   `prestige_score` axis with weight 0.10 (re-balance the others). Confirm
   the WFA jumps in Devon's ranking.

3. **Travel-cost lookup.** Right now `travel_budget_usd` is a single scalar.
   Make it a dict keyed by host city (`{"Boston": 600, "Las Vegas": 400, ...}`)
   and have `score_cost` look up the conference's location.

4. **Two-author submission constraint.** Add a `coauthor_status` field to
   `PaperFit`. If at least one author is a graduate student, the eligibility
   score uses the higher status. Confirm Maya can still submit to FDIC if
   Prof. Gao co-authors.
